# Week 7 — Adding Distance to the Gravity Model

The classic gravity model has two inputs: economic size (GDP) and distance.

We measure the distance from Washington DC to each of the 30 partner countries' capital cities using the Haversine formula, which accounts for Earth's curvature. The result — `distance_km` and `log_distance` — gets added to `master_trade_flow.csv` so the GNN and ST-GCN models can use it as a node feature.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "final").is_dir():
        return cwd
    if (cwd.parent / "data" / "final").is_dir():
        return cwd.parent
    return cwd


BASE = _repo_root()
df = pd.read_csv(BASE / "data/final/master_trade_flow.csv")

# Check both country columns
top30_exp = (
    df.groupby("country_exp")["exports_usd"]
    .sum().nlargest(30).index.tolist()
)
top30_imf = (
    df.groupby("country_imf")["exports_usd"]
    .sum().nlargest(30).index.tolist()
)

print("country_exp names (top 30)")
for c in sorted(top30_exp):
    print(f"  {c}")

print("\ncountry_imf names (top 30)")
for c in sorted(top30_imf):
    print(f"  {c}")

print("\nAre they the same 30 countries?")
print("Only in exp:", sorted(set(top30_exp) - set(top30_imf)))
print("Only in imf:", sorted(set(top30_imf) - set(top30_exp)))

country_exp names (top 30)
  Australia
  Belgium
  Brazil
  Canada
  Chile
  China
  Colombia
  Dominican Republic
  France
  Germany
  Hong Kong
  India
  Ireland
  Israel
  Italy
  Japan
  Korea, South
  Malaysia
  Mexico
  Netherlands
  Saudi Arabia
  Singapore
  Spain
  Switzerland
  Taiwan
  Thailand
  Turkey
  United Arab Emirates
  United Kingdom
  Vietnam

country_imf names (top 30)
  Australia
  Belgium
  Brazil
  Canada
  Chile
  China, People'S Republic Of
  Colombia
  Dominican Republic
  France
  Germany
  Hong Kong Special Administrative Region, People'S Republic Of China
  India
  Ireland
  Israel
  Italy
  Japan
  Korea, Republic Of
  Malaysia
  Mexico
  Netherlands
  Saudi Arabia
  Singapore
  Spain
  Switzerland
  Taiwan Province Of China
  Thailand
  Tã¼Rkiye, Republic Of
  United Arab Emirates
  United Kingdom
  Vietnam

Are they the same 30 countries?
Only in exp: ['China', 'Hong Kong', 'Korea, South', 'Taiwan', 'Turkey']
Only in imf: ["China, People'S Republic Of"

### Compute Distances — Haversine from Washington DC

30 countries, 30 distances. Each entry is the great-circle distance (in km) from DC to the country's capital city. Distance is static — it doesn't change over time — so this only needs to be computed once and then merged into every row for that country.

### Capital Coordinates and Distance Calculation

Latitude and longitude for each of the 30 partner capitals. The Haversine function computes the straight-line distance across the Earth's surface. We also take log(distance) since the gravity model relationship is non-linear — doubling the distance doesn't halve trade, so the log scale fits better.

In [2]:
from math import radians, sin, cos, sqrt, atan2

DC_LAT, DC_LON = 38.9072, -77.0369

capital_coords = {
    "Australia"           : (-35.2809,  149.1300),  # Canberra
    "Belgium"             : ( 50.8503,    4.3517),  # Brussels
    "Brazil"              : (-15.7975,  -47.8919),  # Brasilia
    "Canada"              : ( 45.4215,  -75.6972),  # Ottawa
    "Chile"               : (-33.4569,  -70.6483),  # Santiago
    "China"               : ( 39.9042,  116.4074),  # Beijing
    "Colombia"            : (  4.7110,  -74.0721),  # Bogota
    "Dominican Republic"  : ( 18.4861,  -69.9312),  # Santo Domingo
    "France"              : ( 48.8566,    2.3522),  # Paris
    "Germany"             : ( 52.5200,   13.4050),  # Berlin
    "Hong Kong"           : ( 22.3193,  114.1694),  # Hong Kong
    "India"               : ( 28.6139,   77.2090),  # New Delhi
    "Ireland"             : ( 53.3498,   -6.2603),  # Dublin
    "Israel"              : ( 31.7683,   35.2137),  # Jerusalem
    "Italy"               : ( 41.9028,   12.4964),  # Rome
    "Japan"               : ( 35.6762,  139.6503),  # Tokyo
    "Korea, South"        : ( 37.5665,  126.9780),  # Seoul
    "Malaysia"            : (  3.1390,  101.6869),  # Kuala Lumpur
    "Mexico"              : ( 19.4326,  -99.1332),  # Mexico City
    "Netherlands"         : ( 52.3676,    4.9041),  # Amsterdam
    "Saudi Arabia"        : ( 24.7136,   46.6753),  # Riyadh
    "Singapore"           : (  1.3521,  103.8198),  # Singapore
    "Spain"               : ( 40.4168,   -3.7038),  # Madrid
    "Switzerland"         : ( 46.9481,    7.4474),  # Bern
    "Taiwan"              : ( 25.0330,  121.5654),  # Taipei
    "Thailand"            : ( 13.7563,  100.5018),  # Bangkok
    "Turkey"              : ( 39.9334,   32.8597),  # Ankara
    "United Arab Emirates": ( 24.4539,   54.3773),  # Abu Dhabi
    "United Kingdom"      : ( 51.5074,   -0.1278),  # London
    "Vietnam"             : ( 21.0285,  105.8542),  # Hanoi
}

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return round(R * 2 * atan2(sqrt(a), sqrt(1-a)))

distances = {c: haversine(DC_LAT, DC_LON, lat, lon)
             for c, (lat, lon) in capital_coords.items()}

print("Distances from Washington DC (sorted nearest → farthest):\n")
for country, dist in sorted(distances.items(), key=lambda x: x[1]):
    print(f"  {country:<28} {dist:>7,} km")

Distances from Washington DC (sorted nearest → farthest):

  Canada                           733 km
  Dominican Republic             2,372 km
  Mexico                         3,032 km
  Colombia                       3,814 km
  Ireland                        5,442 km
  United Kingdom                 5,898 km
  Spain                          6,088 km
  France                         6,165 km
  Netherlands                    6,190 km
  Belgium                        6,216 km
  Switzerland                    6,598 km
  Germany                        6,710 km
  Brazil                         6,796 km
  Italy                          7,217 km
  Chile                          8,073 km
  Turkey                         8,724 km
  Israel                         9,496 km
  Saudi Arabia                  10,837 km
  Japan                         10,906 km
  China                         11,146 km
  Korea, South                  11,164 km
  United Arab Emirates          11,346 km
  India          

### Merge Distance into Master Dataset

Mapping the computed distances onto the master trade flow file by country name, then save it back. Every row for a given country now carries the same distance values, ready for the GNN and ST-GCN models to pick up as a node feature.

In [3]:
df = pd.read_csv(BASE / "data/final/master_trade_flow.csv")

# Map using country_exp (clean names)
df["distance_km"] = df["country_exp"].map(distances)
df["log_distance"] = np.log(df["distance_km"])

print("NaN in distance_km:", df["distance_km"].isna().sum())
print("New columns:", ["distance_km", "log_distance"])
print("\nSample (distance sorted):")
print(
    df[["country_exp", "distance_km", "log_distance"]]
    .drop_duplicates("country_exp")
    .sort_values("distance_km")
    .head(10)
    .to_string(index=False)
)

print(f"\nOld shape: {df.shape}")
df.to_csv(BASE / "data/final/master_trade_flow.csv", index=False)
print(f"New shape: {df.shape}")
print("Saved master_trade_flow.csv with distance_km and log_distance")

NaN in distance_km: 16704
New columns: ['distance_km', 'log_distance']

Sample (distance sorted):
       country_exp  distance_km  log_distance
            Canada        733.0      6.597146
Dominican Republic       2372.0      7.771489
            Mexico       3032.0      8.016978
          Colombia       3814.0      8.246434
           Ireland       5442.0      8.601902
    United Kingdom       5898.0      8.682369
             Spain       6088.0      8.714075
            France       6165.0      8.726643
       Netherlands       6190.0      8.730690
           Belgium       6216.0      8.734882

Old shape: (19584, 22)
New shape: (19584, 22)
Saved master_trade_flow.csv with distance_km and log_distance


### Save Distance Reference File and Verify



In [4]:
dist_df = pd.DataFrame(
    list(distances.items()),
    columns=["country", "distance_km"]
).sort_values("distance_km").reset_index(drop=True)

dist_df["log_distance"] = np.log(dist_df["distance_km"])

dist_df.to_csv(BASE / "data/distance_from_dc.csv", index=False)

print("Saved: data/distance_from_dc.csv")
print(f"Rows: {len(dist_df)}")
print(f"\nFull table:")
print(dist_df.to_string(index=False))

Saved: data/distance_from_dc.csv
Rows: 30

Full table:
             country  distance_km  log_distance
              Canada          733      6.597146
  Dominican Republic         2372      7.771489
              Mexico         3032      8.016978
            Colombia         3814      8.246434
             Ireland         5442      8.601902
      United Kingdom         5898      8.682369
               Spain         6088      8.714075
              France         6165      8.726643
         Netherlands         6190      8.730690
             Belgium         6216      8.734882
         Switzerland         6598      8.794522
             Germany         6710      8.811354
              Brazil         6796      8.824089
               Italy         7217      8.884195
               Chile         8073      8.996280
              Turkey         8724      9.073833
              Israel         9496      9.158626
        Saudi Arabia        10837      9.290721
               Japan        10906

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data" / "final").is_dir():
        return cwd
    if (cwd.parent / "data" / "final").is_dir():
        return cwd.parent
    return cwd


FINAL = _repo_root() / "data" / "final"

print("=" * 60)
print("COMPLETE PROJECT VERIFICATION")
print("=" * 60)

# === ANOMALY DETECTION MODELS ===
print("\n--- ANOMALY DETECTION ---\n")

# IF V2
if2 = pd.read_csv(FINAL / "anomaly_top30_results.csv")
print(f"IF V2: {if2['is_anomaly'].sum()} anomalies | {if2['country_imf'].nunique()} countries")

# LSTM
lstm = pd.read_csv(FINAL / "lstm_anomaly_results.csv")
print(f"LSTM:  {lstm['lstm_anomaly'].sum()} anomalies | {lstm['country_display'].nunique()} countries")

# IF vs LSTM overlap
if2_set = set(if2[if2["is_anomaly"]==True].apply(lambda r: f"{r.iloc[0]}_{r['date']}", axis=1))
lstm_set = set(lstm[lstm["lstm_anomaly"]==True].apply(lambda r: f"{r['country_display']}_{r['date']}", axis=1))
overlap = len(if2_set & lstm_set)
print(f"\nIF only:   {len(if2_set) - overlap}")
print(f"LSTM only: {len(lstm_set) - overlap}")
print(f"Both:      {overlap}")

# === FORECASTING MODELS ===
print("\n--- FORECASTING MODELS ---\n")
print(f"{'Model':<25} {'R²':>8} {'MAE':>8} {'RMSE':>8} {'MAPE':>8} {'wMAPE':>8} {'Anomalies':>10}")
print("-" * 85)

forecasting = {
    "XGBoost V2": (FINAL / "xgb_forecast_results.csv", "actual", "predicted", "xgb_anomaly"),
    "XGBoost V3 (COVID)": (FINAL / "xgb_forecast_results_v3.csv", "actual", "predicted", "xgb_anomaly"),
    "GNN Neural Gravity": (FINAL / "gnn_neural_gravity_residuals.csv", "y_log_exports", "pred_log_exports", "gnn_anomaly"),
    "ST-GCN": (FINAL / "stgcn_anomaly_results.csv", "y_log_exports", "pred_log_exports", "stgcn_anomaly"),
}

for name, (path, actual_col, pred_col, anom_col) in forecasting.items():
    df = pd.read_csv(path)
    y_true = df[actual_col].values
    y_pred = df[pred_col].values
    
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    wmape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100
    anomalies = df[anom_col].sum()
    
    print(f"{name:<25} {r2:>8.4f} {mae:>8.4f} {rmse:>8.4f} {mape:>7.2f}% {wmape:>7.2f}% {anomalies:>10}")

# === DATE RANGES ===
print("\n--- DATE RANGES ---\n")
files = {
    "IF V2": FINAL / "anomaly_top30_results.csv",
    "LSTM": FINAL / "lstm_anomaly_results.csv",
    "XGBoost V2": FINAL / "xgb_forecast_results.csv",
    "XGBoost V3": FINAL / "xgb_forecast_results_v3.csv",
    "GNN": FINAL / "gnn_neural_gravity_residuals.csv",
    "ST-GCN": FINAL / "stgcn_anomaly_results.csv",
}

for name, path in files.items():
    df = pd.read_csv(path)
    date_col = "date" if "date" in df.columns else [c for c in df.columns if "date" in c.lower()][0]
    country_col = [c for c in df.columns if "country" in c.lower()][0]
    print(f"{name:<15} {df[date_col].min()} to {df[date_col].max()} | {df[country_col].nunique()} countries | {len(df)} rows")

print("\n" + "=" * 60)
print("USE THESE NUMBERS IN YOUR REPORT AND SLIDES")
print("=" * 60)

COMPLETE PROJECT VERIFICATION

--- ANOMALY DETECTION ---

IF V2: 143 anomalies | 30 countries
LSTM:  152 anomalies | 30 countries

IF only:   121
LSTM only: 130
Both:      22

--- FORECASTING MODELS ---

Model                           R²      MAE     RMSE     MAPE    wMAPE  Anomalies
-------------------------------------------------------------------------------------
XGBoost V2                  0.9707   0.1070   0.1511    0.49%    0.49%         34
XGBoost V3 (COVID)          0.9596   0.1346   0.1829    0.62%    0.62%         79
GNN Neural Gravity          0.9705   0.1119   0.1517    0.52%    0.51%         32
ST-GCN                      0.9691   0.1145   0.1552    0.53%    0.52%         30

--- DATE RANGES ---

IF V2           2017-02-01 to 2024-12-01 | 30 countries | 2850 rows
LSTM            2017-12-01 to 2024-12-01 | 30 countries | 2550 rows
XGBoost V2      2023-01-01 to 2024-12-01 | 30 countries | 720 rows
XGBoost V3      2020-01-01 to 2024-12-01 | 30 countries | 1800 rows
GNN    